In [1]:
import sys; sys.path.insert(0, "..")
from fantasy.yahoo_client import get_my_roster, get_current_matchup, get_roster_by_team, get_league_categories
from fantasy.nba_stats import get_player_stats, get_games_this_week, batch_get_player_stats
from fantasy.analysis import project_team_categories, classify_categories
from fantasy.llm import build_matchup_prompt, ask_gemini
import pandas as pd

In [2]:
import importlib, os
from dotenv import load_dotenv
load_dotenv("/mnt/c/Users/louis/Documents/dev/nba_fantasy/.env", override=True)

import fantasy.llm
importlib.reload(fantasy.llm)
from fantasy.llm import build_matchup_prompt, ask_gemini

In [3]:
matchup = get_current_matchup()
my_roster = get_my_roster()
opp_roster = get_roster_by_team(matchup["opponent_team_key"])
categories = get_league_categories()
week_start, week_end = matchup["week_start"], matchup["week_end"]
print(f"Week {matchup['week']}: {week_start} → {week_end}")
print(f"Opponent team: {matchup['opponent_team_key']}")

[2026-03-24 00:40:19,078 DEBUG] [yahoo_oauth.oauth.__init__] Checking 
[2026-03-24 00:40:19,083 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 3918.4053752422333
[2026-03-24 00:40:19,084 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN HAS EXPIRED
[2026-03-24 00:40:19,085 DEBUG] [yahoo_oauth.oauth.refresh_access_token] REFRESHING TOKEN
[2026-03-24 00:40:19,345 DEBUG] [yahoo_oauth.oauth.token_is_valid] ELAPSED TIME : 0.25908684730529785
[2026-03-24 00:40:19,346 DEBUG] [yahoo_oauth.oauth.token_is_valid] TOKEN IS STILL VALID


Week 22: 2026-03-23 → 2026-03-29
Opponent team: 466.l.26708.t.3


In [4]:
def enrich_players(roster):
    # Fetch stats in parallel, then get game counts
    with_stats = batch_get_player_stats(roster)
    return [{**p, "games_remaining": get_games_this_week(p["team_abbr"], week_start, week_end)}
            for p in with_stats]

my_players = enrich_players(my_roster)
opp_players = enrich_players(opp_roster)

In [5]:
my_proj = project_team_categories(my_players)
opp_proj = project_team_categories(opp_players)

def to_league_cats(proj, cats):
    """Map raw stat projections to Yahoo league scoring category values."""
    result = {}
    for cat in cats:
        if cat in proj:
            result[cat] = proj[cat]
        elif cat == "FG%":
            result[cat] = proj["FGM"] / proj["FGA"] if proj.get("FGA", 0) > 0 else 0.0
        elif cat == "FT%":
            result[cat] = proj["FTM"] / proj["FTA"] if proj.get("FTA", 0) > 0 else 0.0
        elif cat == "3PTM":
            result[cat] = proj.get("FG3M", 0.0)
        elif cat == "ST":
            result[cat] = proj.get("STL", 0.0)
        elif cat == "TO":
            result[cat] = proj.get("TOV", 0.0)
        elif cat == "FGM/A":
            result[cat] = proj.get("FGM", 0.0)
        elif cat == "FTM/A":
            result[cat] = proj.get("FTM", 0.0)
    return result

my_cats = to_league_cats(my_proj, categories)
opp_cats = to_league_cats(opp_proj, categories)
classification = classify_categories(my_cats, opp_cats)

df = pd.DataFrame({
    "My Team": my_cats,
    "Opponent": opp_cats,
    "Status": classification,
}).round(3)
print(df.to_string())

       My Team  Opponent  Status
FGM/A  302.050   306.305  tossup
FG%      0.489     0.483  tossup
FTM/A  120.030   131.882    loss
FT%      0.769     0.802  tossup
3PTM    78.410    99.227    loss
PTS    801.790   843.592    loss
REB    245.210   287.062    loss
AST    200.860   206.892  tossup
ST      51.030    44.700     win
BLK     23.900    29.900    loss
TO      93.730   104.882    loss


In [6]:
prompt = build_matchup_prompt(my_cats, opp_cats, classification)
advice = ask_gemini(prompt)
print("\n=== WEEKLY GAME PLAN ===\n")
print(advice)


=== WEEKLY GAME PLAN ===

Okay, here's the strategy for this week, focusing on maximizing your wins and mitigating your losses based on the projections:

**Overall Strategy:**

We're projected to lose a lot of categories, so let's play smart. We need to **aggressively chase wins in the toss-up categories while solidifying ST (our only projected win).** It looks like 3PTM, PTS, REB, BLK, TO, and FTM/A are going to be very difficult to win. It's unlikely we can overcome those deficits, so don't overextend resources trying to. Let's focus on what's possible and punt the categories we are losing by a wide margin.

**Categories to Target:**

*   **ST (Solidify):** Maintain this lead through smart player selection (high steals rates) and minutes management. This is our rock.
*   **FGM/A, FG%, AST (Swing):** These are the areas where focused effort can make the difference between winning and losing. Close games can come down to just a few shots.
*   **FT%: Punt** While technically a toss-up,

In [7]:
print(prompt)

You are an expert fantasy basketball analyst. Here is this week's H2H category matchup projection:

Category    My Team   Opponent     Status
------------------------------------------
FGM/A         302.0      306.3     tossup
FG%             0.5        0.5     tossup
FTM/A         120.0      131.9       loss
FT%             0.8        0.8     tossup
3PTM           78.4       99.2       loss
PTS           801.8      843.6       loss
REB           245.2      287.1       loss
AST           200.9      206.9     tossup
ST             51.0       44.7        win
BLK            23.9       29.9       loss
TO             93.7      104.9       loss

Provide a concise weekly strategy: which categories to focus on winning, which to punt, and 2-3 specific streaming/lineup actions to swing toss-up categories.


In [8]:

print("=== MY TEAM ===")
for p in my_players:
  proj = p.get("projected", {}) if "projected" in p else {}
  if not proj and p["stats"]:
      from fantasy.analysis import project_player
      proj = project_player(p["stats"], p["games_remaining"], p.get("status", ""))
  print(f"{p['name']:<25} {p['games_remaining']} games  "
        f"PTS:{proj.get('PTS',0):.1f}  REB:{proj.get('REB',0):.1f}  "
        f"AST:{proj.get('AST',0):.1f}  3PM:{proj.get('FG3M',0):.1f}  "
        f"STL:{proj.get('STL',0):.1f}  BLK:{proj.get('BLK',0):.1f}  "
        f"TOV:{proj.get('TOV',0):.1f}")

print("\n=== OPPONENT ===")
for p in opp_players:
  proj = p.get("projected", {}) if "projected" in p else {}
  if not proj and p["stats"]:
      from fantasy.analysis import project_player
      proj = project_player(p["stats"], p["games_remaining"], p.get("status", ""))
  print(f"{p['name']:<25} {p['games_remaining']} games  "
        f"PTS:{proj.get('PTS',0):.1f}  REB:{proj.get('REB',0):.1f}  "
        f"AST:{proj.get('AST',0):.1f}  3PM:{proj.get('FG3M',0):.1f}  "
        f"STL:{proj.get('STL',0):.1f}  BLK:{proj.get('BLK',0):.1f}  "
        f"TOV:{proj.get('TOV',0):.1f}")

=== MY TEAM ===
Jalen Brunson             3 games  PTS:71.6  REB:10.6  AST:23.4  3PM:7.1  STL:2.6  BLK:0.5  TOV:7.5
Dyson Daniels             4 games  PTS:48.0  REB:28.1  AST:22.2  3PM:1.4  STL:9.0  BLK:1.6  TOV:5.2
Jrue Holiday              4 games  PTS:65.2  REB:17.4  AST:24.2  3PM:9.9  STL:3.8  BLK:0.4  TOV:10.9
Jalen Green               2 games  PTS:38.7  REB:7.7  AST:5.7  3PM:4.7  STL:2.3  BLK:0.3  TOV:5.0
Naji Marshall             3 games  PTS:47.4  REB:13.3  AST:12.0  3PM:2.7  STL:2.6  BLK:0.3  TOV:7.2
Kevin Durant              4 games  PTS:103.6  REB:22.6  AST:16.8  3PM:9.3  STL:3.5  BLK:3.3  TOV:12.5
Evan Mobley               3 games  PTS:58.5  REB:28.0  AST:7.9  3PM:2.9  STL:1.8  BLK:5.3  TOV:4.2
Onyeka Okongwu            4 games  PTS:56.1  REB:30.8  AST:13.1  3PM:7.2  STL:4.4  BLK:5.5  TOV:5.1
Jalen Suggs               4 games  PTS:54.6  REB:13.8  AST:19.8  3PM:8.3  STL:6.8  BLK:2.8  TOV:10.4
Tre Jones                 4 games  PTS:56.1  REB:11.9  AST:18.5  3PM:2.7  STL:4.9  